This notebook runs ECMWF's aifs-single-v1 data-driven model, using ECMWF's [open data](https://www.ecmwf.int/en/forecasts/datasets/open-data) dataset and the [anemoi-inference](https://anemoi-inference.readthedocs.io/en/latest/apis/level1.html) package.

## Phase 0 Walkthrough Notes

This copy is project-owned. Dependencies are managed by `uv`, and heavy AIFS downloads/forecast execution are disabled by default so the notebook can run locally as a teaching smoke test. Set `RUN_FULL_AIFS_BASELINE = True` when you are ready to fetch ECMWF open data and run AIFS.

# 1. Install Required Packages and Imports

In [1]:
# Dependencies are managed by uv. Run from the project root when needed:
#   uv sync --extra notebooks --extra aifs --group dev
# Do not install packages from inside this notebook.

In [2]:
import datetime
from collections import defaultdict

import numpy as np
import torch
import earthkit.data as ekd
import earthkit.regrid as ekr

from anemoi.inference.outputs.printer import print_state
from anemoi.inference.runners.simple import SimpleRunner
from ecmwf.opendata import Client as OpendataClient

from weather_research.phase0 import choose_torch_device

DEVICE = choose_torch_device()
RUN_OPEN_DATA_SAMPLE = False
RUN_FULL_AIFS_BASELINE = True
RUN_AIFS_ROLLOUT = False
SAMPLE_PARAM = ["10u"]
print(f"Torch local device preference: {DEVICE}")
print(f"RUN_OPEN_DATA_SAMPLE={RUN_OPEN_DATA_SAMPLE}")
print(f"RUN_FULL_AIFS_BASELINE={RUN_FULL_AIFS_BASELINE}")
print(f"RUN_AIFS_ROLLOUT={RUN_AIFS_ROLLOUT}")


Torch local device preference: mps
RUN_OPEN_DATA_SAMPLE=False
RUN_FULL_AIFS_BASELINE=True
RUN_AIFS_ROLLOUT=False


# 2. Retrieve Initial Conditions from ECMWF Open Data




### List of parameters to retrieve form ECMWF open data

In [3]:
PARAM_SFC = ["10u", "10v", "2d", "2t", "msl", "skt", "sp", "tcw", "lsm", "z", "slor", "sdor"]
PARAM_SOIL =["vsw","sot"]
PARAM_PL = ["gh", "t", "u", "v", "w", "q"]
LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
SOIL_LEVELS = [1,2]

### Select a date

In [4]:
if RUN_OPEN_DATA_SAMPLE or RUN_FULL_AIFS_BASELINE:
    DATE = OpendataClient().latest()
else:
    DATE = datetime.datetime(2024, 1, 1, 0)
    print("Using fixed demo date because no ECMWF data is being downloaded.")


To ensure the stability of our systems and to preserve resources for our operational activities (network, compute, etc.), access to the open-data portal is limited to 500 simultaneous connections. This limit helps us guarantee reliable service for our operational users, especially during periods of high demand. For added reliability, the open-data is replicated across AWS, Azure, and Google Cloud. If you experience difficulties accessing the portal directly, you can also retrieve the data from these cloud platforms.
Some requested dates are before 2026-05-12. Data before this date uses a different stream structure (06/18 UTC runs are under stream=scda/scwv) compared to data on or after 2026-05-12 (where those runs are under stream=oper/wave). This is handled automatically, but be aware that the underlying file structure differs across this boundary.


In [5]:
print("Initial date is", DATE)

Initial date is 2026-05-05 18:00:00


### Get the data from the ECMWF Open Data API

In [6]:
def get_open_data(param, levelist=[]):
    fields = defaultdict(list)
    # Get the data for the current date and the previous date
    for date in [DATE - datetime.timedelta(hours=6), DATE]:
        data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist)
        for f in data:
            # Open data is between -180 and 180, we need to shift it to 0-360
            assert f.to_numpy().shape == (721,1440)
            values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
            # Interpolate the data to from 0.25 to N320
            values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            # Add the values to the list
            name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levelist else f.metadata("param")
            fields[name].append(values)

    # Create a single matrix for each parameter
    for param, values in fields.items():
        fields[param] = np.stack(values)

    return fields

### Get Input Fields

In [7]:
fields = {}

#### Add the single levels fields

In [8]:
if RUN_FULL_AIFS_BASELINE:
    fields.update(get_open_data(param=PARAM_SFC))
elif RUN_OPEN_DATA_SAMPLE:
    fields.update(get_open_data(param=SAMPLE_PARAM))
    sample = fields[SAMPLE_PARAM[0]]
    print(f"Downloaded sample field {SAMPLE_PARAM[0]} with shape {sample.shape}")
    print(f"Sample range: {float(np.nanmin(sample)):.3f} to {float(np.nanmax(sample)):.3f}")
else:
    print("No ECMWF surface fields requested.")


<multiple>:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.


<multiple>:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

In [9]:
if RUN_FULL_AIFS_BASELINE:
    soil = get_open_data(param=PARAM_SOIL, levelist=SOIL_LEVELS)
else:
    soil = {}
    print("Full baseline disabled: soil fields are not downloaded in the one-field sample run.")


<multiple>:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

<multiple>:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Soil parameters have been renamed since training this model, we need to rename to the original names

In [10]:
mapping = {"sot_1": "stl1", "sot_2": "stl2", "vsw_1": "swvl1", "vsw_2": "swvl2"}
for k, v in soil.items():
    fields[mapping[k]] = v
print(f"Mapped {len(soil)} soil fields into AIFS training names.")


Mapped 4 soil fields into AIFS training names.


#### Add the pressure levels fields

In [11]:
if RUN_FULL_AIFS_BASELINE:
    fields.update(get_open_data(param=PARAM_PL, levelist=LEVELS))
else:
    print("Full baseline disabled: pressure-level fields are not downloaded in the one-field sample run.")


<multiple>:   0%|          | 0.00/55.0M [00:00<?, ?B/s]

<multiple>:   0%|          | 0.00/53.9M [00:00<?, ?B/s]

#### Convert geopotential height into geopotential

In [12]:
# Transform geopotential height GH to geopotential Z.
if RUN_FULL_AIFS_BASELINE:
    for level in LEVELS:
        gh = fields.pop(f"gh_{level}")
        fields[f"z_{level}"] = gh * 9.80665
else:
    print("No gh -> z conversion needed for the one-field sample run.")


### Create Initial State

In [13]:
input_state = dict(date=DATE, fields=fields) if fields else None
print("Initial state object built:", input_state is not None)
if input_state is not None:
    print(f"Field count in this run: {len(input_state['fields'])}")
    print(f"Fields present: {sorted(input_state['fields'].keys())[:10]}")

expected_surface = PARAM_SFC
expected_soil = ["swvl1", "swvl2", "stl1", "stl2"]
expected_pressure = [f"{param}_{level}" for param in ["z", "t", "u", "v", "w", "q"] for level in LEVELS]
expected_aifs_fields = expected_surface + expected_soil + expected_pressure
print(f"Full Appendix A/AIFS field count expected later: {len(expected_aifs_fields)}")
print("First 12 expected fields:", expected_aifs_fields[:12])


Initial state object built: True
Field count in this run: 94
Fields present: ['10u', '10v', '2d', '2t', 'lsm', 'msl', 'q_100', 'q_1000', 'q_150', 'q_200']
Full Appendix A/AIFS field count expected later: 94
First 12 expected fields: ['10u', '10v', '2d', '2t', 'msl', 'skt', 'sp', 'tcw', 'lsm', 'z', 'slor', 'sdor']


# 3. Load the Model and Run the Forecast

### Download the Model's Checkpoint from Hugging Face & create a Runner

In [14]:
checkpoint = {"huggingface":"ecmwf/aifs-single-1.1"}

To reduce the memory usage of the model once can set certain environment variables, like the number of chunks of the model's mapper.
Please refer to:
- https://anemoi.readthedocs.io/projects/models/en/latest/modules/layers.html#anemoi-inference-num-chunks
- https://pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf

for more information. To do so, you can use the code below:
```
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['ANEMOI_INFERENCE_NUM_CHUNKS']='16'
```

In [15]:
if RUN_AIFS_ROLLOUT and input_state is not None and len(input_state["fields"]) == len(expected_aifs_fields):
    runner = SimpleRunner(checkpoint, device=DEVICE)
else:
    runner = None
    print("Phase 0 checkpoint: full initial fields are built. AIFS rollout remains disabled until RUN_AIFS_ROLLOUT=True.")


Phase 0 checkpoint: full initial fields are built. AIFS rollout remains disabled until RUN_AIFS_ROLLOUT=True.


**Note - changing the device from GPU to CPU**

- Running the transformer model used on the CPU is tricky, it depends on the FlashAttention library which only supports Nvidia and AMD GPUs, and is optimised for performance and memory usage
- In newer versions of anemoi-models, v0.4.2 and above, there is an option to switch off flash attention and uses Pytorchs Scaled Dot Product Attention (SDPA). The code snippet below shows how to overwrite a model from a checkpoint to use SDPA. Unfortunately it's not optimised for memory usage in the same way, leading to much greater memory usage. Please refer to https://github.com/ecmwf/anemoi-inference/issues/119 for more details

#### Run the forecast

In [16]:
state = None
if runner is not None:
    for state in runner.run(input_state=input_state, lead_time=24):
        print_state(state)
else:
    print("Phase 0 completed the IC-building step. The next AIFS execution step is +24h rollout with lead_time=24.")


Phase 0 completed the IC-building step. The next AIFS execution step is +24h rollout with lead_time=24.


**Note**
Due to the non-determinism of GPUs, users will be unable to exactly reproduce an official AIFS forecast when running AIFS Single themselves.
If you want to enforece determinism at GPU level, you can do so enforcing the following settings:

```
#First in your terminal
export CUBLAS_WORKSPACE_CONFIG=:4096:8

#And then before running inference:
import torch
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

```
Using the above approach will significantly increase runtime. Additionally, the input conditions come from open data, which we reproject from o1280 (the original projection of IFS initial conditions) to n320 (AIFS resolution) by first converting them to a 0.25-degree grid. In the operational setup, however, data is reprojected directly from o1280 to n320. This difference in reprojection methods may lead to variations in the resulting input conditions, causing minor differences in the forecast.

# 4. Inspect the generated forecast

#### Plot a field

In [17]:
# Plotting dependencies are managed by uv via the notebooks extra.

In [18]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as tri

In [19]:
def fix(lons):
    # Shift the longitudes from 0-360 to -180-180.
    return np.where(lons > 180, lons - 360, lons)

if state is None:
    print("No forecast plot for this Phase 0 checkpoint because the notebook stopped after IC construction.")
else:
    latitudes = state["latitudes"]
    longitudes = state["longitudes"]
    values = state["fields"]["100u"]

    fig, ax = plt.subplots(figsize=(11, 6), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=":")

    triangulation = tri.Triangulation(fix(longitudes), latitudes)
    contour = ax.tricontourf(triangulation, values, levels=20, transform=ccrs.PlateCarree(), cmap="RdBu")
    fig.colorbar(contour, ax=ax, orientation="vertical", shrink=0.7, label="100u")
    plt.title(f"100m winds (100u) at {state['date']}")
    plt.show()


No forecast plot for this Phase 0 checkpoint because the notebook stopped after IC construction.
